# 第 7 章: cinema データの探索と可視化

特徴量と興行収入の関係、外れ値、線形回帰の予測誤差を確認する。

In [ ]:
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import seaborn as sns
from japanese_font import use_japanese_font

from lib.chapter02.iris_preprocessing import (
    column_means,
    fill_missing,
    split_train_test,
)
from lib.chapter07.cinema_regression import (
    FEATURES,
    TARGET,
    fit_linear_regression,
    load_cinema,
    prepare_cinema,
    r2_score,
    remove_outliers,
)
from lib.dataset import data_dir

use_japanese_font();

In [ ]:
df = load_cinema(data_dir() / "cinema.csv")
df.isna().sum()

In [ ]:
corr = df[FEATURES + [TARGET]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1);

In [ ]:
corr[TARGET].drop(TARGET).sort_values(ascending=False).round(3)

In [ ]:
outliers = df.drop(remove_outliers(df).index)
fig, axes = plt.subplots(1, len(FEATURES), figsize=(16, 4), sharey=True)
for ax, column in zip(axes, FEATURES, strict=True):
    ax.scatter(df[column], df[TARGET], alpha=0.6)
    ax.scatter(outliers[column], outliers[TARGET], color="red", label="外れ値")
    ax.set_xlabel(column)
axes[0].set_ylabel(TARGET)
axes[0].legend();

In [ ]:
split = prepare_cinema(data_dir() / "cinema.csv", test_size=0.2, seed=0)
model = fit_linear_regression(split.x_train, split.t_train)
y = model.predict(split.x_test)

low, high = split.t_test.min(), split.t_test.max()
plt.scatter(split.t_test, y)
plt.plot([low, high], [low, high], color="gray", linestyle="--")
plt.xlabel("実測値")
plt.ylabel("予測値");

In [ ]:
residual = split.t_test.to_numpy() - y
plt.scatter(y, residual)
plt.axhline(0, color="gray", linestyle="--")
plt.xlabel("予測値")
plt.ylabel("残差（実測値 - 予測値）");

In [ ]:
def evaluate(train, test):
    means = column_means(train, FEATURES)
    model = fit_linear_regression(fill_missing(train[FEATURES], means), train[TARGET])
    y = model.predict(fill_missing(test[FEATURES], means))
    return {
        "SNS2 の係数": round(model.coefficients["SNS2"], 3),
        "テストデータの R2": round(r2_score(test[TARGET], y), 4),
    }


same_split = split_train_test(df, df[TARGET], test_size=0.2, seed=0)
{
    "外れ値を残して学習": evaluate(same_split.x_train, same_split.x_test),
    "外れ値を除いて学習": evaluate(
        remove_outliers(same_split.x_train), same_split.x_test
    ),
}

In [ ]:
scores = {}
for seed in range(5):
    s = prepare_cinema(data_dir() / "cinema.csv", test_size=0.2, seed=seed)
    y = fit_linear_regression(s.x_train, s.t_train).predict(s.x_test)
    scores[seed] = round(r2_score(s.t_test, y), 4)
scores